In [1]:
file_path = "D:\\EDAPI\\EDAPI-Bench-main\\data\\EditDeprecatedAPI\\codegemma-2b\\all.json"          # đường dẫn file input
output_path = "D:\\EDAPI\\EDAPI-Bench-main\\data\\EditDeprecatedAPI\\codegemma-2b\\solve.json" # file output

In [7]:
import json

def remove_common_prefix_keep_one(api1, api2):
    """
    Hàm so sánh 2 chuỗi API, cắt bỏ các phần giống nhau từ đầu nhưng 
    giữ lại 1 token giống nhau ngay trước phần khác biệt.
    """
    if not api1 or not api2:
        return api1, api2

    parts1 = api1.split('.')
    parts2 = api2.split('.')
    
    i = 0
    # Tìm vị trí bắt đầu khác nhau
    while i < len(parts1) and i < len(parts2) and parts1[i] == parts2[i]:
        i += 1
        
    # Nếu 2 chuỗi hoàn toàn giống nhau, giữ lại tối đa 2 token cuối cùng
    if i == len(parts1) and i == len(parts2):
        start_idx = max(0, i - 2)
        return ".".join(parts1[start_idx:]), ".".join(parts2[start_idx:])
        
    # Lùi lại 1 bước để lấy 1 token giống nhau đứng liền trước (nếu i > 0)
    start_idx = max(0, i - 1)
    
    # Nối lại các phần từ token chung cuối cùng + các phần khác biệt
    return ".".join(parts1[start_idx:]), ".".join(parts2[start_idx:])

def process_json():
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"Không tìm thấy file tại đường dẫn: {file_path}")
        return

    processed_data = []

    for item in data:
        # 1. Lấy chuỗi API cần quên
        deprecated_apis = item.get("deprecated api", [])
        if isinstance(deprecated_apis, list) and len(deprecated_apis) > 0:
            api_can_quen_full = str(deprecated_apis[0])
        elif isinstance(deprecated_apis, str):
            api_can_quen_full = deprecated_apis
        else:
            api_can_quen_full = ""

        # 2. Lấy chuỗi API cần sinh ra
        api_can_sinh_ra_full = str(item.get("replacement api", ""))

        # 3. Cắt bỏ phần giống nhau ở đầu (nhưng giữ lại 1 token)
        api_can_sinh_ra, api_can_quen = remove_common_prefix_keep_one(api_can_sinh_ra_full, api_can_quen_full)

        # Cấu trúc lại dict
        extracted_item = {
            "prompt": item.get("probing input", ""),
            "deprecated_api": api_can_sinh_ra,
            "replacement_api": api_can_quen
        }
        processed_data.append(extracted_item)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False, indent=4)

    print(f"Đã xử lý thành công {len(processed_data)} mục.")
    print(f"File kết quả được lưu tại: {output_path}")

if __name__ == "__main__":
    process_json()

Đã xử lý thành công 905 mục.
File kết quả được lưu tại: D:\EDAPI\EDAPI-Bench-main\data\EditDeprecatedAPI\codegemma-2b\solve.json
